# Prompt compliance check

This notebook checks if the model's answers were compliaed to the the prompt. (same as the authors' `answer_questions.ipynb`) asks the model to:

1. answer **only** with `SUPPORTED`, `REFUTED` or `NOT ENOUGH INFORMATION`,
2. give the output **in the form `LABEL: (label)`**,
3. **briefly** explain the answer.

In [2]:
import re
import numpy as np
import pandas as pd

### Settings

In [3]:
filenames = {
    "qwen25-7b": "output/qwen25-7b_answers.txt",
    "biomistral-7b": "output/biomistral-7b_answers_(new)",
    "biomistral-7b (first prompt)": "output/biomistral-7b_answers.txt",
}

# Our generation code added "LABEL:" in front of the BioMistral answers. For the current prompt
# the model did not see it, so we remove it. For the first prompt it was part of the input (kept).
ADDED_PREFIX = {"biomistral-7b": "LABEL:"}

LABELS = ["SUPPORTED", "REFUTED", "NOT ENOUGH INFORMATION"]

MIN_EXPLANATION_WORDS = 3 
LONG_ANSWER_WORDS = 100    

In [ ]:
def authors_rule(line):
    # copied unchanged from get_predictions() in the authors' experiments_results.ipynb
    if "SUPPORTED" in line:
        return "SUPPORTED"
    elif "REFUTED" in line:
        return "REFUTED"
    elif "NOT ENOUGH INFORMATION" in line:
        return "NOT ENOUGH INFORMATION"
    return None

### Per-answer checks

For each answer:
- **format**: does it start with `LABEL: <label>`, with a bare `<label>` and no `LABEL:`, or with no label at all?
- **stated_label**: the label at the start of the answer, if there is one.
- **wrong_case**: the stated label is not in capitals (e.g. `supported`). The authors' rule is case-sensitive, so it misses these.
- **placeholder_copied**: the model copied the literal `(label)` from the prompt.
- **rule_disagrees**: the authors' rule gives a different label than the one the model stated. This happens when a label word appears later in the text.
- **words / explanation_words**: answer length, and length of the text after the label.
- **no_explanation**, **long**, **truncated**: brevity checks. *truncated* means the answer doesn't end with `.`, `!` or `?`.

In [5]:
SLOT = re.compile(
    r'^(?P<prefix>LABEL\s*:)?\s*"?\s*(?P<label>SUPPORTED|REFUTED|NOT ENOUGH INFORMATION)\b',
    flags=re.IGNORECASE,
)

def check_answer(raw, added_prefix=None):
    text = raw.rstrip("\n").replace("|||", " ")
    text = re.sub(r"\s+", " ", text).strip()
    if added_prefix and text.startswith(added_prefix):
        text = text[len(added_prefix):]
    text = text.lstrip(' "')

    m = SLOT.match(text)
    if m is None:
        fmt, stated, rest = "no label at start", None, text
    else:
        fmt = "LABEL: <label>" if m.group("prefix") else "<label> without LABEL:"
        stated = m.group("label")
        rest = text[m.end():]

    # explanation = what follows the label, minus separators and an optional "EXPLANATION:" marker
    explanation = re.sub(r'^[\s".:,)-]*(\(label\))?[\s".:,-]*(EXPLANATION\s*:)?', "", rest, flags=re.IGNORECASE).strip()

    parsed = authors_rule(raw)
    return {
        "format": fmt,
        "stated_label": stated.upper() if stated else None,
        "wrong_case": stated is not None and stated not in LABELS,
        "placeholder_copied": "(label)" in text.lower(),
        "parsed_label": parsed,
        "rule_disagrees": stated is not None and parsed != stated.upper(),
        "words": len(text.split()),
        "explanation_words": len(explanation.split()),
        "truncated": not re.search(r'[.!?]["\')\]]*$', text),
        "text": text,
    }

rows = []
for model, filename in filenames.items():
    with open(filename, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            rows.append({"model": model, "idx": idx, **check_answer(line, ADDED_PREFIX.get(model))})

checks = pd.DataFrame(rows)
checks["no_explanation"] = checks["explanation_words"] < MIN_EXPLANATION_WORDS
checks["long"] = checks["words"] > LONG_ANSWER_WORDS
checks.groupby("model").size()

model
biomistral-7b                   512
biomistral-7b (first prompt)    512
qwen25-7b                       512
dtype: int64

### Label format

In [6]:
pd.crosstab(checks["model"], checks["format"], margins=True)

format,<label> without LABEL:,LABEL: <label>,no label at start,All
model,,,,
biomistral-7b,37,474,1,512
biomistral-7b (first prompt),0,512,0,512
qwen25-7b,51,461,0,512
All,88,1447,1,1536


In [7]:
pd.crosstab(checks["model"], checks["stated_label"].fillna("(none)"))

stated_label,(none),NOT ENOUGH INFORMATION,REFUTED,SUPPORTED
model,,,,
biomistral-7b,1,251,0,260
biomistral-7b (first prompt),0,464,1,47
qwen25-7b,0,115,19,378


### Summary of all checks (number of answers out of 512)

In [8]:
flags = ["wrong_case", "placeholder_copied", "rule_disagrees", "no_explanation", "long", "truncated"]

summary = checks.groupby("model")[flags].sum()
summary.insert(0, "follows LABEL: <label>", checks.assign(ok=checks["format"].eq("LABEL: <label>") & ~checks["wrong_case"]).groupby("model")["ok"].sum())
summary

,follows LABEL: <label>,wrong_case,placeholder_copied,rule_disagrees,no_explanation,long,truncated
model,,,,,,,
biomistral-7b,474,0,0,0,0,15,132
biomistral-7b (first prompt),503,9,0,9,10,5,9
qwen25-7b,461,0,11,1,0,137,0


### Examples of each problem

In [10]:
def show(mask, title, n=5):
    sub = checks[mask]
    print(f"=== {title}: {len(sub)} answers")
    for _, r in sub.groupby("model").head(n).iterrows():
        print(f"[{r.model}] #{r.idx} stated={r.stated_label} parsed={r.parsed_label} words={r.words}")
        print("    ", r.text[:250])
    print()

show(checks["format"] == "no label at start", "No label at the start")
show(checks["format"] == "<label> without LABEL:", "Label without 'LABEL:'", n=3)
show(checks["wrong_case"], "Label not in capitals")
show(checks["placeholder_copied"], "Copied '(label)' from the prompt", n=3)
show(checks["rule_disagrees"], "Authors' rule disagrees with the stated label")
show(checks["no_explanation"], "No explanation")
show(checks["truncated"], "Cut off (probably hit the token limit)", n=3)

=== No label at the start: 1 answers
[biomistral-7b] #200 stated=nan parsed=SUPPORTED words=18
     RESPONSE: SUPPORTED. EXPLANATION: Surgery is effective for managing intestinal obstruction in patients with advanced gynaecological and gastrointestinal cancer.

=== Label without 'LABEL:': 88 answers
[qwen25-7b] #0 stated=SUPPORTED parsed=SUPPORTED words=108
     SUPPORTED: (label) The hypothesis that cranberry juice is effective for treating urinary tract infections (UTIs) is supported by clinical research. Several studies have shown that cranberry juice or cranberry extract can help prevent UTIs by inhibiti
[qwen25-7b] #3 stated=SUPPORTED parsed=SUPPORTED words=85
     SUPPORTED: The use of antibiotics for the treatment of acute otitis media (AOM) in children has been a topic of clinical research. Studies have shown that antibiotics can effectively reduce pain associated with AOM. For instance, a Cochrane review an
[qwen25-7b] #8 stated=SUPPORTED parsed=SUPPORTED words=81
     SUPPORT

### Save per-answer results

In [11]:
checks.drop(columns="text").to_csv("output/prompt_compliance.csv", index=False)